# 🔧 Notebook 02 — Feature Engineering & Model Training
**FMCG Promotional Analytics | Portfolio Project**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Panosemmanouilidis2/fmcg-ds-technical-portfolio/blob/main/promotional-analytics/notebooks/02_feature_engineering.ipynb)

This notebook encodes features, trains six model families, tunes the champion XGBoost model, and saves all artefacts.

| | |
|---|---|
| Input | `sample_synthetic.csv` loaded directly from GitHub |
| Champion model | XGBoost (tuned) |
| Outputs | `models/` — trained model, feature list, medians |
| Next | `03_model_training.ipynb` (Vertex AI deployment) |


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 1 — INSTALL & IMPORT
# ══════════════════════════════════════════════════════════════════
!pip install lightgbm shap --quiet

import pandas as pd
import numpy as np
import json, pickle, os, warnings
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import RandomizedSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import lightgbm as lgb
warnings.filterwarnings('ignore')

os.makedirs('models', exist_ok=True)
os.makedirs('data',   exist_ok=True)
print("✅ Imports OK")


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 2 — LOAD DATA (from GitHub — no upload needed)
# ══════════════════════════════════════════════════════════════════
CSV_URL = "https://raw.githubusercontent.com/Panosemmanouilidis2/fmcg-ds-technical-portfolio/main/promotional-analytics/notebooks/sample_synthetic.csv"

df = pd.read_csv(CSV_URL)
df['Market'] = df['Market'].map({
    'Western Europe' : 'Market A',
    'Southeast Asia' : 'Market B'
})

print(f"Loaded: {len(df):,} rows × {df.shape[1]} cols")
print("Markets:", df['Market'].value_counts().to_dict())


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 3 — DEFINE TARGET & LEAKAGE COLUMNS
# Leakage columns only exist AFTER a promotion runs.
# Including them would let the model cheat — it would learn from
# information a planner cannot know before execution.
# ══════════════════════════════════════════════════════════════════
TARGET = 'ActualSalesVolume'

LEAKAGE_COLS = [
    TARGET,
    'ForecastAccuracyRatio',  # derived from actual — post-execution
]

print(f"Target        : {TARGET}")
print(f"Leakage cols  : {LEAKAGE_COLS}")
print(f"Feature pool  : {df.shape[1] - len(LEAKAGE_COLS)} columns")


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 4 — ENCODE CATEGORICAL COLUMNS
# One-hot encoding for low-cardinality columns (≤15 unique values):
#   Each category gets its own binary column so the model can learn
#   the independent effect of each promotion type, market etc.
# Label encoding for higher-cardinality columns:
#   Converts strings to integers — efficient for tree-based models.
# ══════════════════════════════════════════════════════════════════
df_enc = df.copy()

OHE_COLS = ['Market', 'PromotionType', 'InStoreLocation', 'Division']
df_enc = pd.get_dummies(df_enc, columns=OHE_COLS, drop_first=False)
print(f"After one-hot encoding: {df_enc.shape[1]} columns")

LE_COLS = ['Customer', 'Category', 'PromotionMonth']
encoders = {}
for col in LE_COLS:
    le = LabelEncoder()
    df_enc[col] = le.fit_transform(df_enc[col].astype(str))
    encoders[col] = le
    print(f"  Label-encoded: {col}  ({list(le.classes_)})")

pickle.dump(encoders, open('models/label_encoders.pkl', 'wb'))
print("\n✅ Encoders saved → models/label_encoders.pkl")


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 5 — VERIFY ALL COLUMNS ARE NUMERIC
# ══════════════════════════════════════════════════════════════════
remaining_obj = df_enc.select_dtypes(include='object').columns.tolist()
if remaining_obj:
    print(f"⚠️  Still object dtype: {remaining_obj}")
else:
    print("✅ No object columns — all encoded")
print(f"Null values : {df_enc.isnull().sum().sum()}")
print(f"Total cols  : {df_enc.shape[1]}")


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 6 — LOG TRANSFORM TARGET
# np.log1p(x) = log(x+1) handles zero values safely.
# Reversed at inference time with np.expm1().
# ══════════════════════════════════════════════════════════════════
df_enc['LogTarget'] = np.log1p(df_enc[TARGET])

print("Original target:")
print(f"  Mean : {df_enc[TARGET].mean():>10,.0f}  skew={df_enc[TARGET].skew():.2f}")
print("Log-transformed:")
print(f"  Mean : {df_enc['LogTarget'].mean():>10.4f}  skew={df_enc['LogTarget'].skew():.2f}")


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 7 — CHRONOLOGICAL TRAIN / TEST SPLIT  (80/20)
# Sorted by in-store start month to preserve time order.
# Shuffling would allow the model to see "future" promotions when
# predicting "past" ones — a form of data leakage that inflates
# performance metrics without reflecting real-world accuracy.
# ══════════════════════════════════════════════════════════════════
df_sorted = df_enc.sort_values('InstoreStartDate_Month').reset_index(drop=True)

feature_cols = [c for c in df_sorted.columns
                if c not in LEAKAGE_COLS + ['LogTarget']]

X = df_sorted[feature_cols]
y = df_sorted['LogTarget']

split_idx   = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Train : {len(X_train):,} rows  ({split_idx/len(X)*100:.0f}%)")
print(f"Test  : {len(X_test):,} rows  ({(1-split_idx/len(X))*100:.0f}%)")
print(f"Features: {len(feature_cols)}")

json.dump(feature_cols, open('models/feature_cols.json', 'w'))
print("\n✅ Feature list saved → models/feature_cols.json")


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 8 — SCALE FEATURES FOR LINEAR MODELS
# StandardScaler: each column → mean=0, std=1.
# Fitted on training data only — applying test statistics to the
# fit would leak test information into training.
# Tree-based models (RF, XGBoost, LightGBM) do not need scaling.
# ══════════════════════════════════════════════════════════════════
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

pickle.dump(scaler, open('models/scaler.pkl', 'wb'))
print("✅ Scaler saved → models/scaler.pkl")


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 9 — TRAIN ALL 6 MODELS
# Evaluating 6 model families avoids defaulting to one algorithm
# without evidence it outperforms simpler alternatives.
# ══════════════════════════════════════════════════════════════════
def evaluate(name, y_true, y_pred):
    return {
        'Model': name,
        'R²'   : round(r2_score(y_true, y_pred), 4),
        'MAE'  : round(mean_absolute_error(y_true, y_pred), 4),
        'RMSE' : round(np.sqrt(mean_squared_error(y_true, y_pred)), 4),
    }

results = []

lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
results.append(evaluate('Linear Regression', y_test, lr.predict(X_test_scaled)))

ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train)
results.append(evaluate('Ridge', y_test, ridge.predict(X_test_scaled)))

lasso = Lasso(alpha=0.01)
lasso.fit(X_train_scaled, y_train)
results.append(evaluate('Lasso', y_test, lasso.predict(X_test_scaled)))

rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
results.append(evaluate('Random Forest', y_test, rf.predict(X_test)))

xgb_base = xgb.XGBRegressor(n_estimators=200, random_state=42,
                              objective='reg:squarederror', verbosity=0)
xgb_base.fit(X_train, y_train)
results.append(evaluate('XGBoost (baseline)', y_test, xgb_base.predict(X_test)))

lgb_model = lgb.LGBMRegressor(n_estimators=200, random_state=42, verbose=-1)
lgb_model.fit(X_train, y_train)
results.append(evaluate('LightGBM', y_test, lgb_model.predict(X_test)))

results_df = pd.DataFrame(results).sort_values('R²', ascending=False)
print(results_df.to_string(index=False))


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 10 — HYPERPARAMETER TUNING (XGBoost champion)
# Two-stage approach:
#   Stage 1 — search on 2K sample for speed
#   Stage 2 — retrain winner on full training set for accuracy
# ══════════════════════════════════════════════════════════════════
param_grid = {
    'n_estimators'    : [200, 400, 600],
    'max_depth'       : [3, 5, 7],
    'learning_rate'   : [0.01, 0.05, 0.1],
    'subsample'       : [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0],
    'reg_alpha'       : [0, 0.1, 1.0],
    'reg_lambda'      : [1.0, 5.0, 10.0],
}

sample_size = min(2000, len(X_train))
search = RandomizedSearchCV(
    xgb.XGBRegressor(objective='reg:squarederror', verbosity=0, random_state=42),
    param_grid, n_iter=30, cv=3, scoring='r2', random_state=42, n_jobs=-1
)
search.fit(X_train.iloc[:sample_size], y_train.iloc[:sample_size])
print(f"Best params   : {search.best_params_}")
print(f"CV R² (sample): {search.best_score_:.4f}")


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 11 — RETRAIN ON FULL TRAINING SET WITH BEST PARAMS
# ══════════════════════════════════════════════════════════════════
xgb_tuned = xgb.XGBRegressor(
    **search.best_params_,
    objective='reg:squarederror',
    verbosity=0, random_state=42
)
xgb_tuned.fit(X_train, y_train)
y_pred_tuned = xgb_tuned.predict(X_test)

r2_test   = r2_score(y_test, y_pred_tuned)
mae_test  = mean_absolute_error(y_test, y_pred_tuned)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_tuned))

print("=== TUNED XGBoost — TEST SET ===")
print(f"  R\u00b2   : {r2_test:.4f}")
print(f"  MAE  : {mae_test:.4f}")
print(f"  RMSE : {rmse_test:.4f}")

# ── Overfitting check ────────────────────────────────────────────────────────
# A model that scores perfectly on training data but poorly on test data
# has memorised the training set rather than learned real patterns.
# Healthy gap: < 0.05 R2 points. Gap > 0.15 = red flag.
r2_train = r2_score(y_train, xgb_tuned.predict(X_train))
gap      = r2_train - r2_test

print(f"\n=== OVERFITTING CHECK ===")
print(f"  Train R2 : {r2_train:.4f}")
print(f"  Test  R2 : {r2_test:.4f}")
print(f"  Gap      : {gap:.4f}  (overfit warning if gap > 0.15)")
if gap > 0.15:
    print("  \u26a0\ufe0f  Possible overfit — consider more regularisation")
else:
    print("  \u2705 OK — model generalises well")


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 12 — SAVE ALL ARTEFACTS
# ══════════════════════════════════════════════════════════════════
booster = xgb_tuned.get_booster()
booster.save_model('models/xgb_tuned.json')
print("✅ Model saved → models/xgb_tuned.json")

eval_df = X_test.copy()
eval_df['ActualSalesVolume']    = np.expm1(y_test.values)
eval_df['PredictedSalesVolume'] = np.expm1(y_pred_tuned)
eval_df['LogActual']            = y_test.values
eval_df['LogPredicted']         = y_pred_tuned
eval_df.to_csv('data/eval_results.csv', index=False)
print("✅ Eval results saved → data/eval_results.csv")

medians = {col: float(X_train[col].median()) for col in feature_cols}
json.dump(medians, open('models/feature_medians.json', 'w'))
print("✅ Feature medians saved → models/feature_medians.json")

print("\n=== ALL MODEL ARTEFACTS ===")
for f in sorted(os.listdir('models')):
    kb = os.path.getsize(f'models/{f}') / 1024
    print(f"  models/{f:<40} {kb:>7.1f} KB")

print("\n▶ Next: 03_model_training.ipynb  OR  04_model_evaluation_shap.ipynb")


## 📝 Training Summary

| Metric | Value |
|---|---|
| Champion model | XGBoost (tuned via RandomizedSearchCV) |
| Train/test split | Chronological 80/20 — no data leakage |
| Target transformation | log1p → reversed with expm1 at inference |
| Saved artefacts | `xgb_tuned.json`, `feature_cols.json`, `feature_medians.json`, `scaler.pkl`, `label_encoders.pkl` |

**Next step:** `03_model_training.ipynb` (deploy) or `04_model_evaluation_shap.ipynb` (evaluate)
